# Alpha Momentum: live paper-trading notebook

Concentrated equity momentum, a.k.a. Alpha Chaser (see `research/alpha_research.ipynb`).

**Run All** once a month after month end. With TWS (or IB Gateway) open, account value and positions are read automatically through a **read-only** API session; orders are entered manually.

Credentials and the account id are read from `ibkr_keys.env` (not in the repository, see `ibkr_keys.env.example`).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.append(os.path.join("..", "src"))
from data import download_prices, prepare_monthly
from strategies import trend_momentum_signal, alpha_chaser_weights
from execution import apply_overlay, rebalance_orders
from ibkr import load_keys, read_portfolio
from factsheet import build_factsheet

STRATEGY   = "ALPHA"
DATA_DIR   = os.path.join("..", "data")
KEYS       = load_keys(os.path.join("..", "ibkr_keys.env"))
ACCOUNT    = KEYS.get("IBKR_ACCOUNT_ALPHA", "")
USE_IBKR   = True     # False: type positions manually in section 3
IBKR_PORT  = 7497     # 7497 TWS paper | 7496 TWS live | 4002 Gateway paper | 4001 Gateway live
CLIENT_ID  = 12       # any integer, different for each notebook

## 1. Data and model weights

In [ ]:
UNIVERSE = ["SPY", "QQQ", "SMH", "XLK", "XLY", "IWM", "ARKK", "SHV"]

prices = download_prices(UNIVERSE, years=20, cache_path=os.path.join(DATA_DIR, "prices_alpha.csv"))
md = prepare_monthly(prices, sma_window=100)
weights = alpha_chaser_weights(trend_momentum_signal(md), md.mom_3m, top_n=2, buffer=0.10, max_weight=0.50)

## 2. Targets (optional discretionary overlay)
Tweaks must net to zero. Every deviation from the model is logged in the journal line.

In [ ]:
tweaks = {t: 0.0 for t in UNIVERSE}
journal = "Model weights, no discretionary change."

targets = apply_overlay(weights.iloc[-1], tweaks, journal)

## 3. Current positions

In [ ]:
# Backtest ticker -> EUR-listed UCITS ETF actually traded
UCITS_MAP = {
    "SPY":  {"ticker": "SXR8", "isin": "IE00B5BMR087", "desc": "iShares Core S&P 500"},
    "QQQ":  {"ticker": "SXRV", "isin": "IE00B53SZB19", "desc": "iShares Nasdaq 100"},
    "SMH":  {"ticker": "SEC0", "isin": "IE000I8KRLL9", "desc": "iShares MSCI Semiconductors"},
    "XLK":  {"ticker": "QDVE", "isin": "IE00B3WJKG14", "desc": "iShares S&P 500 IT Sector"},
    "XLY":  {"ticker": "IUCD", "isin": "IE00B4MCHD36", "desc": "iShares S&P 500 Cons. Discr."},
    "IWM":  {"ticker": "ZPRR", "isin": "IE00BJ38QD84", "desc": "SPDR Russell 2000"},
    "ARKK": {"ticker": "ARKK", "isin": "IE000GA3D489", "desc": "ARK Innovation UCITS ETF"},
    "SHV":  {"ticker": "IBCC", "isin": "IE00BGSF1X88", "desc": "iShares USD Treasury 0-1yr"},
}

if USE_IBKR:
    account_value, holdings = read_portfolio(ACCOUNT, UCITS_MAP, port=IBKR_PORT, client_id=CLIENT_ID)
else:
    account_value = 0.0                                  # e.g. 105480.00
    holdings = {k: 0.0 for k in UCITS_MAP}              # current value in EUR per ticker

## 4. Order list

In [ ]:
orders = rebalance_orders(targets, holdings, account_value, UCITS_MAP, min_trade=50.0)

## 5. Factsheet

In [ ]:
START_DATE     = "2026-05-01"
FUND_NAME      = "Alpha Momentum Portfolio"
UNIVERSE_LABEL = "7 equity ETFs + T-bills"
ASSET_CLASS = {"SPY": "US large caps", "QQQ": "US tech / growth", "SMH": "Semiconductors",
               "XLK": "US tech / growth", "XLY": "Consumer discretionary", "IWM": "US small caps",
               "ARKK": "Disruptive innovation", "SHV": "Cash-like (T-bills)"}
STRATEGY_TEXT = """
Concentrated equity momentum strategy aiming to capture the strongest trends in high-beta US
equity segments. The universe covers 7 ETFs: S&P 500, Nasdaq 100, semiconductors, technology,
consumer discretionary, small caps and disruptive innovation. The portfolio is rebalanced monthly.
An ETF is eligible only if its price is above its 100-day moving average and its 3-month return is
positive. The two eligible ETFs with the strongest 3-month momentum are held at 50% each; current
holdings receive a 10-point momentum bonus to limit unnecessary turnover. If only one ETF is
eligible it takes 50% and the rest goes to 0-1Y US Treasuries; if none is eligible the portfolio is
fully in T-bills. Implemented with EUR-listed UCITS ETFs.
"""

build_factsheet(
    strategy=STRATEGY, account=ACCOUNT, account_value=account_value, holdings=holdings,
    targets=targets, model_weights=weights, ucits_map=UCITS_MAP, asset_class=ASSET_CLASS,
    fund_name=FUND_NAME, universe=UNIVERSE_LABEL, strategy_text=STRATEGY_TEXT, start_date=START_DATE,
    flex_token=KEYS.get("FLEX_TOKEN_ALPHA", ""),
    flex_query_id=KEYS.get("FLEX_QUERY_ID_ALPHA", ""),
    use_ibkr=USE_IBKR, data_dir=DATA_DIR, out_dir=os.path.join("..", "factsheets"),
);